In [0]:
from pyspark.sql.functions import col, round

df = spark.table("retail_project_dbricks.default.bronze_retail_raw")

print("Bronze Rows:", df.count())

In [0]:
df_silver = (
    df
    .dropna(subset=["Invoice", "StockCode", "CustomerID"])
    .filter(col("Quantity") > 0)
    .filter(col("Price") > 0)
    .filter(~col("Invoice").startswith("C"))
    .withColumn(
        "Revenue",
        round(col("Quantity") * col("Price"), 2)
    )
)

In [0]:
print("Silver Rows:", df_silver.count())

display(df_silver.limit(10))

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "retail_project_dbricks.default.silver_retail_clean"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM retail_project_dbricks.default.silver_retail_clean
""").show()